# Рендер Blender в Google Colab

Этот ноутбук рендерит уже подготовленный проект Blender без изменения его сцены, камер, движка или настроек качества.

**Перед загрузкой:** если в сцене используются внешние текстуры, HDRI, кеши симуляций или другие файлы, либо упакуйте их в Blender через `File → External Data → Pack Resources`, либо загрузите ZIP-архив с папкой проекта. Внутри ZIP должен быть `.blend` и сохранена исходная структура папок.

Для Cycles включите в Colab: `Runtime → Change runtime type → T4 GPU` (или другой GPU).

In [ ]:
# Выберите версию, которая не старее версии Blender, в которой сохранён ваш файл.
# При необходимости замените значение, например на '4.3.2'.
BLENDER_VERSION = '4.2.0'

# True: попытаться включить GPU для сцен Cycles.
# Для Eevee параметр не нужен: Eevee использует GPU автоматически.
ENABLE_CYCLES_GPU = True

# 'ANIMATION' — все кадры из диапазона, сохранённого в .blend.
# 'STILL' — один кадр, номер задаётся в STILL_FRAME.
RENDER_MODE = 'ANIMATION'
STILL_FRAME = 1

# При необходимости дополнительно скачивать результат на компьютер из последней ячейки.
DOWNLOAD_RESULT = False

In [ ]:
# Установка Blender. Явная версия делает запуск воспроизводимым.
import os
import subprocess
from pathlib import Path

blender_series = '.'.join(BLENDER_VERSION.split('.')[:2])
archive_name = f'blender-{BLENDER_VERSION}-linux-x64.tar.xz'
archive_url = (
    f'https://download.blender.org/release/Blender{blender_series}/{archive_name}'
)
install_root = Path('/content/blender')
blender_dir = install_root / f'blender-{BLENDER_VERSION}-linux-x64'
BLENDER = blender_dir / 'blender'

if not BLENDER.exists():
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'libgl1', 'libglib2.0-0', 'libsm6'], check=True)
    install_root.mkdir(parents=True, exist_ok=True)
    subprocess.run(['wget', '--show-progress', '-O', str(install_root / archive_name), archive_url], check=True)
    subprocess.run(['tar', '-xf', str(install_root / archive_name), '-C', str(install_root)], check=True)

subprocess.run([str(BLENDER), '--version'], check=True)

In [ ]:
# Загрузите один .blend или один ZIP с проектом.
from google.colab import files
import shutil
import zipfile

workspace = Path('/content/blender_project')
shutil.rmtree(workspace, ignore_errors=True)
workspace.mkdir()

uploaded = files.upload()
if not uploaded:
    raise RuntimeError('Файл не загружен.')

for name, data in uploaded.items():
    uploaded_path = workspace / Path(name).name
    uploaded_path.write_bytes(data)
    if uploaded_path.suffix.lower() == '.zip':
        with zipfile.ZipFile(uploaded_path) as archive:
            archive.extractall(workspace)

blend_files = sorted(workspace.rglob('*.blend'))
if len(blend_files) != 1:
    raise RuntimeError(
        f'Найдено .blend-файлов: {len(blend_files)}. Загрузите ровно один проект.'
    )

BLEND_FILE = blend_files[0]
print(f'Будет отрендерен: {BLEND_FILE}')

In [ ]:
# Подключите Google Drive и задайте папку для результатов.
# Colab покажет ссылку для авторизации доступа к вашему Google Drive.
from google.colab import drive
from datetime import datetime

drive.mount('/content/drive')
DRIVE_OUTPUT_DIR = Path('/content/drive/MyDrive/Blender Renders')
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Рендеры будут сохранены в Google Drive: {DRIVE_OUTPUT_DIR}')

In [ ]:
# Включаем GPU для Cycles, если он доступен. Скрипт ничего не меняет в исходном .blend.
gpu_script = Path('/content/configure_cycles_gpu.py')
gpu_script.write_text('''import bpy
if not bpy.context.preferences.addons.get("cycles"):
    print("Cycles addon is unavailable; using saved render settings.")
else:
    prefs = bpy.context.preferences.addons["cycles"].preferences
    selected = False
    for backend in ("OPTIX", "CUDA"):
        try:
            prefs.compute_device_type = backend
            prefs.get_devices()
            gpu_devices = [device for device in prefs.devices if device.type == "GPU"]
            if gpu_devices:
                for device in prefs.devices:
                    device.use = device.type == "GPU"
                for scene in bpy.data.scenes:
                    if scene.render.engine == "CYCLES":
                        scene.cycles.device = "GPU"
                print(f"Cycles GPU backend: {backend}")
                selected = True
                break
        except Exception as error:
            print(f"{backend} is unavailable: {error}")
    if not selected:
        print("No Cycles GPU detected; Cycles will render on CPU.")
''', encoding='utf-8')

# Отдельная папка на каждый запуск не даёт перезаписать прошлый рендер.
render_name = f'{BLEND_FILE.stem}_{datetime.now():%Y-%m-%d_%H-%M-%S}'
output_dir = DRIVE_OUTPUT_DIR / render_name
output_dir.mkdir()
output_prefix = output_dir / 'frame_'

command = [str(BLENDER), '-b', str(BLEND_FILE)]
if ENABLE_CYCLES_GPU:
    command += ['-P', str(gpu_script)]
command += ['-o', str(output_prefix)]
if RENDER_MODE.upper() == 'STILL':
    command += ['-f', str(STILL_FRAME)]
elif RENDER_MODE.upper() == 'ANIMATION':
    command += ['-a']
else:
    raise ValueError("RENDER_MODE должен быть 'STILL' или 'ANIMATION'.")

print('Команда:', ' '.join(command))
subprocess.run(command, check=True)

In [ ]:
# Результат уже сохранён в Google Drive. При DOWNLOAD_RESULT = True
# эта ячейка дополнительно скачает файл/ZIP на компьютер.
rendered_files = [path for path in output_dir.rglob('*') if path.is_file()]
if not rendered_files:
    raise RuntimeError('Рендер завершился, но файлов в папке результата нет.')

print(f'Готово: {len(rendered_files)} файл(ов) в Google Drive: {output_dir}')

if DOWNLOAD_RESULT:
    if len(rendered_files) == 1:
        result = rendered_files[0]
    else:
        result_base = DRIVE_OUTPUT_DIR / output_dir.name
        shutil.make_archive(str(result_base), 'zip', output_dir)
        result = result_base.with_suffix('.zip')
    print(f'Скачивается: {result.name} ({result.stat().st_size / 1024 / 1024:.1f} MB)')
    files.download(str(result))